# v14 — Full Corpus Cluster with MACRO-TOPIC KEYWORD EXTRACTION
### Lord of the Tensors — CSE Thesis
**UMAP + HDBSCAN / KMeans / Agglomerative | Post-HDBSCAN K re-clustering | 2017–2026**

**Changes from original:**
- Fixed missing comma bug in sinhala_stops
- Consolidated stopwords into single deduplicated list
- Fixed sw_arg dead code / stopwords.json ignored
- Optimized keyword extraction (single global vectorizer + matrix slicing)
- Added stratified silhouette sampling
- Memory cleanup after alignment
- Added UMAP checkpoint saves
- Added min_samples to HDBSCAN config

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import json
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'artifacts').exists():
            return candidate
    raise RuntimeError('Could not locate project root with configs/ and artifacts/.')


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import kneighbors_graph
from hdbscan import HDBSCAN
from umap import UMAP

os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() or 4)

# ── Local Paths ───────────────────────────────────────────────────────────────
BASE_DIR = find_project_root()
EMB_DIR    = BASE_DIR / 'embeddings'
DATA_CSV = BASE_DIR / 'artifacts' / 'final_v14' / 'all_speakers.csv'
SW_PATH = BASE_DIR / 'configs' / 'stopwords.json'
OUTPUT_DIR = BASE_DIR / 'artifacts' / 'final_v14'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'BASE_DIR   : {BASE_DIR}')
print(f'EMB_DIR    : {EMB_DIR}')
print(f'DATA_CSV   : {DATA_CSV}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MIN_CLUSTER_SIZE = 10    # HDBSCAN min_cluster_size
MIN_SAMPLES      = 5     # HDBSCAN min_samples (controls noise sensitivity)
TOP_N_WORDS      = 50    # Keywords shown per cluster
RANDOM_STATE     = 42
SILHOUETTE_SAMPLE = 5000 # Max points for silhouette approximation

## 1. Load Embeddings

In [ ]:
year_files = sorted(
    f for f in EMB_DIR.glob('20*.npy')
    if 'master' not in f.name
)
print(f'Found {len(year_files)} year files:')

year_embs = {}
for f in year_files:
    yr  = int(f.stem)
    arr = np.load(f)
    year_embs[yr] = arr
    print(f'  {f.name}: {arr.shape}')

## 2. Load Metadata & Align

In [ ]:
df_raw = pd.read_csv(DATA_CSV)
df_raw['year'] = pd.to_datetime(df_raw['date'], errors='coerce').dt.year

print('CSV year counts:')
print(df_raw['year'].value_counts().sort_index().to_string())

print('\nPer-year alignment:')
aligned_embs = []
aligned_dfs  = []
skipped      = []

for yr in sorted(year_embs.keys()):
    emb_yr = year_embs[yr]
    df_yr  = df_raw[df_raw['year'] == yr].reset_index(drop=True)
    if len(df_yr) == len(emb_yr):
        aligned_embs.append(emb_yr)
        aligned_dfs.append(df_yr)
        print(f'  [OK]   {yr}: {len(df_yr):>5} speeches')
    else:
        skipped.append(yr)
        print(f'  [SKIP] {yr}: CSV={len(df_yr)} vs emb={len(emb_yr)} — mismatch')

embeddings = np.vstack(aligned_embs)
df         = pd.concat(aligned_dfs, ignore_index=True)

print(f'\nIncluded : {sorted(set(year_embs.keys()) - set(skipped))}')
print(f'Skipped  : {skipped}')
print(f'Final    : {len(df)} speeches,  emb shape {embeddings.shape}')
assert len(df) == len(embeddings), 'Row count mismatch!'
print('Row counts match ✓')

# Free per-year data — no longer needed
del year_embs, aligned_embs, aligned_dfs, df_raw
gc.collect()
print('Memory freed ✓')

## 3. UMAP Dimensionality Reduction

In [ ]:
umap5_path = OUTPUT_DIR / 'umap5.npy'
umap2_path = OUTPUT_DIR / 'umap2.npy'

if umap5_path.exists() and umap2_path.exists():
    print('Loading cached UMAP projections ...')
    emb5 = np.load(umap5_path)
    emb2 = np.load(umap2_path)
    assert len(emb5) == len(embeddings), f'Cached UMAP5 rows ({len(emb5)}) != embeddings ({len(embeddings)}). Delete cache and re-run.'
    print(f'  emb5: {emb5.shape},  emb2: {emb2.shape}')
else:
    print('Fitting UMAP 5D (for clustering) ...')
    umap5 = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                 metric='cosine', random_state=RANDOM_STATE)
    emb5  = umap5.fit_transform(embeddings)
    np.save(umap5_path, emb5)
    print(f'  Done — shape: {emb5.shape}  (saved to {umap5_path.name})')

    print('Fitting UMAP 2D (for visualization) ...')
    umap2 = UMAP(n_neighbors=15, n_components=2, min_dist=0.1,
                 metric='cosine', random_state=RANDOM_STATE)
    emb2  = umap2.fit_transform(embeddings)
    np.save(umap2_path, emb2)
    print(f'  Done — shape: {emb2.shape}  (saved to {umap2_path.name})')

## 4. HDBSCAN Clustering (Primary)

In [ ]:
print(f'Running HDBSCAN (min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}) ...')
hdb = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    metric='euclidean',
    cluster_selection_method='eom'
)
hdb_labels = hdb.fit_predict(emb5)

K_hdbscan = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
noise_n   = (hdb_labels == -1).sum()
noise_pct = 100 * noise_n / len(hdb_labels)
print(f'HDBSCAN  ->  K = {K_hdbscan} clusters')
print(f'             noise = {noise_n} pts  ({noise_pct:.1f}%)')

## 5. KMeans at K = K_hdbscan

In [ ]:
print(f'Running KMeans (K={K_hdbscan}, n_init=20) ...')
km_labels = KMeans(
    n_clusters=K_hdbscan,
    random_state=RANDOM_STATE,
    init='k-means++',
    n_init=20
).fit_predict(emb5)
print('Done ✓')

## 6. Agglomerative Clustering at K = K_hdbscan

In [ ]:
print('Building connectivity graph ...')
connectivity = kneighbors_graph(emb5, n_neighbors=15, include_self=False)

print(f'Running Agglomerative (K={K_hdbscan}, linkage=ward) ...')
agg_labels = AgglomerativeClustering(
    n_clusters=K_hdbscan,
    linkage='ward',
    connectivity=connectivity
).fit_predict(emb5)
print('Done ✓')

## 7. Metrics Summary

In [ ]:
gc.collect()


def silhouette_stratified(emb, labels, max_total=5000, rng_seed=42):
    """Stratified silhouette sampling — preserves cluster proportions."""
    arr  = np.array(labels)
    mask = arr != -1
    if mask.sum() < 2 or len(set(arr[mask])) < 2:
        return 0.0

    emb_valid = emb[mask]
    lab_valid = arr[mask]
    n_valid   = len(lab_valid)

    if n_valid <= max_total:
        return round(float(silhouette_score(emb_valid, lab_valid)), 4)

    # Stratified sampling
    rng = np.random.RandomState(rng_seed)
    unique_labels, counts = np.unique(lab_valid, return_counts=True)
    proportions = counts / counts.sum()
    sample_counts = np.maximum(1, (proportions * max_total).astype(int))

    indices = []
    for lab, n_sample in zip(unique_labels, sample_counts):
        cluster_idx = np.where(lab_valid == lab)[0]
        chosen = rng.choice(cluster_idx, size=min(n_sample, len(cluster_idx)), replace=False)
        indices.extend(chosen)

    indices = np.array(indices)
    return round(float(silhouette_score(emb_valid[indices], lab_valid[indices])), 4)


def cluster_metrics(name, labels, emb):
    arr       = np.array(labels)
    k_found   = len(set(arr)) - (1 if -1 in arr else 0)
    noise_pct = round(100 * (arr == -1).sum() / len(arr), 1)
    valid     = arr != -1
    n_valid   = valid.sum()

    sil = silhouette_stratified(emb, arr, max_total=SILHOUETTE_SAMPLE)
    ch  = round(calinski_harabasz_score(emb[valid], arr[valid]), 2) if n_valid > 1 and len(set(arr[valid])) > 1 else 0
    db  = round(davies_bouldin_score(emb[valid], arr[valid]), 4)    if n_valid > 1 and len(set(arr[valid])) > 1 else 0

    return {
        'Algorithm'         : name,
        'K'                 : k_found,
        'Noise %'           : noise_pct,
        'N Evaluated'       : int(n_valid),
        'Silhouette'        : sil,
        'Calinski-Harabasz' : ch,
        'Davies-Bouldin'    : db,
    }


rows = [
    cluster_metrics('HDBSCAN',       hdb_labels, emb5),
    cluster_metrics('KMeans',        km_labels,  emb5),
    cluster_metrics('Agglomerative', agg_labels, emb5),
]
df_metrics = pd.DataFrame(rows).set_index('Algorithm')
print(df_metrics.to_string())
print('\nNote: HDBSCAN metrics exclude noise points. N Evaluated column shows comparable sample sizes.')

## 8. Stopwords (Consolidated & Deduplicated)

In [ ]:
# ── Sinhala stopwords ─────────────────────────────────────────────────────────
sinhala_stops = [
    # Pronouns & particles
    "මම", "අපි", "අප", "මා", "මගේ", "අපේ", "අපගේ", "ඔහු", "ඇය", "ඔවුන්",
    "ඔවුන්ගේ", "එය", "ඔබ", "ඔය",
    # Demonstratives
    "ඒ", "මේ", "එක", "ඒක", "මේක", "ඒවා", "මේවා", "එක්", "එකක්", "එකට",
    "එකේ", "ඒකට", "ඒකෙ", "ඒකේ", "ඒකෙන්", "එකෙන්", "ඒවාට", "මේකට",
    "එකත්", "මේකයි", "ඒකයි",
    # Conjunctions & postpositions
    "සහ", "හා", "ද", "ට", "ගේ", "වල", "වලට", "වලින්", "වලදී",
    "සඳහා", "ගැන", "මත", "යටතේ", "හරහා", "තුළ", "තුළින්", "මගින්", "මඟින්",
    "සමග", "සමඟ", "විසින්", "අනුව", "පිළිබඳ", "පිළිබඳව", "පිළිබඳවත්",
    "වෙත", "වෙතින්", "වෙතට", "වෙනුවෙන්", "වෙනුවට", "එරෙහිව",
    "සම්බන්ධ", "සම්බන්ධව", "සම්බන්ධයෙන්", "අදාළ", "අදාළව",
    # Verbs (common auxiliaries & light verbs)
    "කර", "කරන", "කරන්න", "කරන්නේ", "කරන්නට", "කරන්නත්", "කරන්නම්",
    "කරනවා", "කරලා", "කළා", "කළ", "කළේ", "කරපු", "කරමින්", "කරමු",
    "කිරීම", "කිරීමට", "කිරීමේ", "කිරීමක්", "කිරීමේදී", "කරනු", "කරගන්න",
    "කරගෙන",
    "කියලා", "කියන", "කියනවා", "කියන්න", "කියන්නේ", "කියන්නම්", "කියා",
    "කියපු", "කියලායි", "කිව්වේ", "කිව්වා",
    "ලබා", "ලබන", "ලද", "ලැබී", "ලැබුණා", "ලැබුණු", "ලැබුණේ", "ලැබෙන",
    "ලැබෙනවා", "ලැබිලා", "ලැබීම",
    "දෙන", "දෙන්න", "දෙන්නේ", "දෙනවා", "දුන්නා", "දුන්නේ", "දීම", "දීමට",
    "දීමේ", "දීලා", "දීපු",
    "ගන්න", "ගන්නවා", "ගන්නට", "ගත්තා", "ගත්තේ", "ගත්", "ගත්ත",
    "ගත්තාම", "ගත්තොත්", "ගත", "ගැනීම", "ගැනීමට", "ගැනීමේ", "ගනිමින්",
    "ගෙන", "ගෙනැල්ලා", "ගෙනෙන්න",
    "වෙනවා", "වෙලා", "වෙන්න", "වෙන්නේ", "වෙමින්", "වෙයි", "වන", "වනවා",
    "වන්නේ", "වී", "වීම", "වීමට", "වූ", "වුණා", "වුණේ", "වුණාට", "වුණාම",
    "වුණු", "වුණොත්", "විය",
    "තිබෙනවා", "තිබෙන", "තිබෙන්නේ", "තිබෙන්න", "තිබුණා", "තිබුණු",
    "තිබුණේ", "තිබුණත්", "තියෙනවා", "තිබෙනවාද",
    "ඉන්න", "ඉන්නේ", "ඉන්නවා", "ඉඳලා",
    "යනවා", "යන්න", "යන්නේ", "යනු",
    "එන", "එන්න", "එන්නේ", "එනවා",
    "ගියා", "ගියේ", "ගිහිල්ලා", "ගිහින්",
    "ආවා", "ආවේ", "ආපු", "ඇවිල්ලා",
    "හදනවා", "හදලා", "හදන", "හදන්නේ", "හදා",
    "දමන්න", "දමා", "දාලා",
    "බලන්න", "බලා", "බලාගෙන",
    "ගහලා", "පවතිනවා", "සලකා",
    # Common adverbs & adjectives
    "ඉතා", "ඉතාම", "ඉතින්", "බොහෝ", "බොහොම", "විශාල", "ලොකු", "පොඩි",
    "හොඳ", "හොඳයි", "හරි", "හරියට", "විශේෂ", "විශේෂයෙන්ම", "ප්‍රධාන",
    "වැදගත්", "අලුත්", "වෙනස්", "සම්පූර්ණයෙන්", "සම්පූර්ණයෙන්ම",
    "දැඩි", "සාධාරණ", "සිරස්", "සුළු", "විවිධ", "සියලු", "සෑම",
    # Temporal
    "අද", "ඊයේ", "හෙට", "දැන්", "දැනට", "දැනටමත්", "පසුගිය", "පස්සේ",
    "පසුව", "පසු", "කලින්", "ඉදිරි", "ඉදිරියේදී", "ඊළඟ", "තවම", "තව",
    "තවත්", "තවදුරටත්", "දිගටම", "නැවත", "නැවතත්", "දින", "දවස්", "දවසේ",
    "මාස", "කාලය", "කාලයක්", "කාලයේ", "කාල", "අවුරුදු", "අවුරුද්දේ",
    "වෙලාවට", "වෙලාවේ", "එදා", "එවකට", "වනකොට", "වෙනකොට",
    # Negation
    "නැහැ", "නෑ", "නැති", "නැතිව", "නැතුව", "බැහැ", "බැරි", "නොවෙයි",
    "නොවන", "නොහැකි", "නොමැති", "කිසිම", "කිසි", "කිසිදු",
    # Discourse markers & fillers
    "තමයි", "නේ", "හැබැයි", "මොකද", "එහෙම", "එතකොට",
    "ඇත්ත", "ඇත්තටම", "මොනවාද", "මොකක්ද", "මොකක්", "මොන",
    "කොහොමද", "ඔව්", "නැද්ද", "පුළුවන්ද", "එසේ", "මෙහෙම",
    # Parliamentary address forms
    "ගරු", "මහතා", "මන්ත්‍රීතුමා", "තුමා", "කථානායකතුමනි", "කථානායක",
    "කථා", "ඔබතුමා", "කථානායකතුමා", "අමාත්‍යතුමා", "ඇමතිතුමා",
    "ඇමතිතුමනි", "මන්ත්‍රීවරයා", "මන්ත්‍රීතුමනි", "නියෝජ්‍ය",
    "පාර්ලිමේන්තුව", "සභාපතිතුමා", "සභාපතිතුමනි", "මහත්මයා",
    "මහත්මිය", "ඔබතුමන්ලා", "ඔබතුමන්ලාට", "ඔබතුමාට",
    "එතුමා", "එතුමාගේ", "එතුමාට", "එතුමන්ලා","මැතිතුමා,",
    "තමුන්නාන්සේලා", "මූලාසනාරූඪ", "තුමනි",
    "සභාව", "සභාවේ", "සභාවට", "සභා", "සභාගත",
    # Other high-frequency function words
    "එතැන", "එහි", "මෙය", "මෙම", "මෙහි", "යන", "එම", "අතර", "මෙලෙස",
    "මෙසේ", "ලෙස", "වශයෙන්", "වශයෙන්ම", "ආකාරයට", "ආකාරය",
    "හැටියට", "විධියට", "පමණ", "පමණයි", "තරම්",
    "ඉහළ", "අඩු", "අවම", "උපරිම", "වඩා",
    "ඇත", "වේ", "ය", "ම", "ක්", "ක", "එ", "ද්",
    "ශ්‍රී", "ලබ", "සිට", "දී", "මහා", "මහ", "විට", "ඇති", "සිදු",
    "බව", "බවට", "බවෙන්", "බවත්", "නිසා", "නිසාවෙන්", "නිසායි",
    "නම්", "නමුත්", "එනමුත්", "එහෙත්", "මිස", "හැර", "තොරව",
    "ඇතුළු", "ඇතුළේ", "ඇතුළත්", "දක්වා", "පටන්", "තෙක්", "තුරු", "තුරා",
    "පවා", "වත්", "විනා", "පරිදි", "සේක",
    "එනිසා", "එබැවින්", "බැවින්", "හෙයින්", "හේතුව", "හේතුවෙන්", "හේතු",
    "අන්න", "ඔන්න", "මෙන්න", "උදෙසා", "පිණිස",
    "වනාහි", "කලී", "කිම", "මන්ද", "පතා", "හනික",
    "බොහෝ", "වහා", "සැනින්", "හෙවත්", "නොහොත්",
    "මෙයින්", "තුලින්", "ගානෙ", "නිතර", "යලි", "පුන",
    # Misc high-frequency
    "කටයුතු", "කටයුත්ත", "රටේ", "රට", "දේවල්", "දේ", "දෙයක්",
    "ඉදිරිපත්", "අවශ්‍ය", "අවශ්‍යයි", "වැඩ", "වැඩක්",
    "ක්‍රියා", "ක්‍රියාත්මක", "රාජ්‍ය", "යම්", "යම්කිසි",
    "කාරණය", "කාරණාව", "කාරණා", "කාරණයක්", "කාරක",
    "කරුණු", "සඳහන්", "තත්ත්වය", "තත්ත්වයක්", "තත්ත්වයට",
    "අවස්ථාව", "අවස්ථාවක්", "අවස්ථාවේ", "අවස්ථාවේදී", "අවස්ථා",
    "නියෝජනය", "ආරම්භ", "සකස්", "ප්‍රකාශ", "සිද්ධ",
    "පත්", "පුළුවන්", "පුළුවන්කම", "හැකි", "හැකියාව",
    "වාගේ", "වාගේම", "වගේම", "වගේ",
    "අයට", "අය", "අයගේ", "කෙනෙක්", "දෙනා", "දෙනෙක්", "දෙනාම",
    "ගොල්ලන්", "කවුරු", "කවුරුත්", "කවදාවත්",
    "අවසන්", "අවසාන", "සියයට", "ප්‍රමාණයක්", "ගණනාවක්", "කිහිපයක්",
    "තුනක්", "දෙක", "පළමු", "පළමුවැනි", "දෙවැනි",
    "පෙනී", "පෙළ", "ගණනේ", "කෙටි", "භාවිත", "හඳුන්වා", "හඳුනා",
    "එකඟ", "අනුමත", "අත්සන්",
    "සතුටු", "අයත්", "තේරුම්", "දැනුවත්", "දැනගන්න", "දැනුම්", "දැන",
    "බලාපොරොත්තු", "උත්සාහ", "සාමාන්‍යයෙන්",
    "මතය", "මතකයි", "මතක", "හිතනවා", "හිතන්නේ", "හිතන",
    "කැමතියි", "කැමැතියි", "පිළිගන්නවා",
    "ඉල්ලා", "එකතු", "ඉවත්", "අනෙක්",
    "එල්ල", "කඩා", "පිට", "අත්", "බල",
    "දිය", "ගැටලුවක්", "වෙනම", "වහාම",
    "සියල්ල", "ඔක්කෝම", "ඔක්කොම",
    "යුතුයි", "යුතු", "යුතුව", "යුත්තේ",
    "එක්ක", "දන්නවා", "දන්නේ", "දන්නා",
    "හිටපු", "හිටියා", "හිටියේ", "සිටින", "සිටි", "සිටිනවා", "සිටියා",
    "ඕනෑ", "ඕනෑම", "එපා",
    "දිහා", "පැත්තෙන්", "පැත්තකින්",
    "ටිකක්", "ඔප්පු", "වතාවක්", "කළාම", "කළාට", "කළත්",
    "අනුගමනය", "තහවුරු", "මුලින්ම", "ක්‍රමයක්", "පැවැති",
    "සහිත", "ඉක්මනින්", "ආසන්න", "කොටසක්",
    "නිකුත්", "මැදිහත්", "තිස්සේ", "නතර", "දිගින්",
    "දීර්ඝ", "මෙයයි", "අහිමි", "ඉෂ්ට", "එළියට",
    "තීන්දුවක්", "වගකීමක්", "හම්බ", "ඉන්",
    "සූදානම්", "අනිවාර්යයෙන්ම", "උදාහරණයක්", "තීරණයක්",
    "අපිත්", "අපිට", "අපට", "මට", "එයට",
    "කොට", "විතර", "විතරක්", "විතරයි",
    "අර", "යහ", "පළ", "යි", "දා", "ළඟ", "පැය", "කල්",
    "වෙන්", "වෙන", "වෙනත්", "මීට", "ඊට",
    "අවස්ථාවේ", "එහි",
    "විරුද්ධ", "විරුද්ධව",
    "මුහුණ", "ගමන්", "දරන", "ලැබෙන",
    "අහන්න", "අහනවා", "අහන්නේ",
    "මතක", "පමණයි", "බලා", "ගෙවන්න", "උත්තර", "කට",
    "නිසි", "සිරස්", "අවධානය", "මෙතැන",
    "යෝජනාව", "පනත", "පිළිතුර", "ප්‍රශ්නය", "විවාදය", "විනාඩි",
    # Interjections
    "අහා", "ආහ්", "ආ", "ඕහෝ", "අනේ", "අඳෝ", "අපොයි", "අපෝ", "අයියෝ",
    "ආයි", "ඌයි", "චී", "චිහ්", "චික්", "හෝ", "දෝ", "දෝහෝ",
    # Comparatives
    "මෙන්", "සේ", "වැනි", "බඳු", "වන්", "අයුරු", "අයුරින්",
    "වඩාත්ම", "නිති", "පමණින්", "අරබයා", "වස්",
    # Additional misc from original lists
    "කාරණා", "අමතරව", "ගෙවන්න", "යොමු", "නැත්නම්", "සමහර", "ටික",
    "හැම", "ගිය", "මාස", "එවැනි", "නොවෙයි", "අවුරුදු",
]

# ── Tamil stopwords ────────────────────────────────────────────────────────────
tamil_stops = [
    # Pronouns
    "ஒரு", "என்று", "இந்த", "அவர்", "என்பது", "என", "நான்", "நாம்",
    "நீங்கள்", "அவர்கள்", "இது", "அது", "அந்த", "இல்", "க்கு", "ஆக",
    "உள்ள", "இல்லை", "மேலும்", "பற்றி", "தான்", "வந்து", "கொண்டு",
    "உள்ளது", "ஆகிய", "என்ன", "எனது", "எனவும்", "அதன்", "அவ்வாறு",
    "இங்கு", "இருந்து", "இருக்கும்", "இருக்கின்றது", "உள்ளனர்",
    "உள்ளார்", "உள்ளன", "எங்கு", "எப்போது", "எப்படி", "எனவே",
    "என்பதை", "என்பதன்", "ஏன்", "ஏற்கனவே", "ஏதேனும்", "ஏனெனில்",
    "ஓர்", "ஓரு", "கொண்ட", "கொண்டே", "கொண்டும்", "கொண்டுள்ள",
    "சில", "சுமார்", "தனது", "தனி", "நமது", "நமக்கு", "நமதுடைய",
    "போது", "போல்", "மட்டும்", "மற்ற", "மற்றும்", "மற்றவை",
    "மற்றவர்கள்", "மற்றவர்", "வந்த", "வந்தது", "வந்தனர்",
    "வந்தான்", "வந்தாள்", "வந்தேன்",
    # Parliamentary
    "கௌரவ", "சபாநாயகர்", "அவர்களே", "அமைச்சர்", "உறுப்பினர்", "திரு",
    "திருமதி", "ஐயா", "பாராளுமன்ற", "மன்றம்", "சபை", "சபைத்தலைவர்",
    "மசோதா", "சட்டம்", "கேள்வி", "பதில்", "விவாதம்", "நேரம்",
    # Common particles & connectors
    "தாங்கள்", "காணொளி", "தகவல்", "பலவேறு", "இடம்பெற்ற",
    "எச்சரிக்கை", "குறைந்த", "தகவல்கள்", "தொடர்பான", "நடவடிக்கை",
    "விவரம்", "காணப்படுகின்றன", "கொல்லப்பட்டனர்", "ஏற்பட்டது",
    "வழியாக", "பாதுகாப்பு", "நாடு", "செய்த", "பகுதி", "கொலை",
    "சென்று", "தற்போது", "ஊடகம்", "அதிகப்படியான", "விலை",
    "ஒன்றாக", "அமைச்சு", "வெளியில்", "செய்தி", "வெளியிடப்பட்டது",
    "விவரங்கள்", "மூலமாக", "விவரமான", "கூறப்பட்டது", "நிலைகள்",
    "செயலாளர்", "வெளியிடப்பட்டுள்ளன", "மூலம்", "கூறினார்",
    "வழங்கினார்", "செய்யப்பட்டுள்ளது", "அங்குள்ள", "அடிப்படையில்",
    "உங்களுடைய", "கேட்டுக்", "இருந்த", "செய்து", "அதாவது", "போன்ற",
    "முடியும்", "அவர்களுக்கு", "விரும்புகின்றேன்", "என்னுடைய",
    "இருக்கின்றார்கள்", "விடயம்", "தொடர்பில்", "இந்தத்",
    "வேண்டுமென", "இருக்கிறது", "நிலையில்", "அந்தக்", "பல்வேறு",
    "வேண்டும்", "நாங்கள்", "கடந்த", "இந்தச்", "அல்லது", "ஆனால்",
    "அரசாங்கம்", "அரசியல்", "அங்கு", "என்ற", "எமது", "எங்களுடைய",
    "தமிழ்", "மக்கள்", "நாட்டில்", "இன்று", "பல", "இருக்கின்ற",
    "இந்தப்", "மிகவும்", "வடக்கு", "நாட்டின்", "குறிப்பாக", "வகையில்",
    "தலைமைதாங்கும்", "ஆகவே", "முஸ்லிம்", "அந்தப்", "நன்றி",
    "மக்களின்", "மக்களுக்கு", "பிரதேச", "மிக", "ஆண்டு", "நாட்டிலே",
    "அவர்களுடைய", "ரூபாய்", "இருக்கின்றன", "பொருளாதார", "முடியாது",
    "காரணமாக", "நிதி", "சனாதிபதி", "வரவு", "மக்களுடைய", "செய்ய",
    "இன்னும்", "தேசிய", "பாரிய", "அபிவிருத்தி", "இலங்கை", "மாண்புமிகு",
    "இவ்வாறான", "கிழக்கு", "மீண்டும்", "காலத்தில்", "மாகாண",
    "கொள்கின்றேன்", "தங்களுடைய", "அரச", "ஜனாதிபதி", "இன்றைய",
    "இரண்டு", "மாவட்டத்தில்", "காணப்படுகின்றது", "மாவட்ட",
    "மட்டக்களப்பு", "புத்தளம்", "இருந்தார்", "தலைவர்", "டாக்டர்",
    "சிறந்த", "உறுப்பினராக", "அனுதாபப்", "ஒருவர்", "இருந்தது",
    "தேசியக்", "கட்சி", "அனுதாபத்தைத்", "பின்னர்", "லங்கா", "எஸ்",
    "அமைச்சராக", "தன்னுடைய", "காங்கிரஸ்", "ஸ்ரீ", "ஸ்ரீ லங்கா",
    "கட்சியின்", "மலையக", "திருத்தச்", "தீர்வு", "சர்வதேச", "நீதி",
    "விடயங்கள்", "சிங்கள", "ரணில்", "நாட்டை", "சபையில்", "நாட்டு",
    "நாட்டினுடைய", "பாராளுமன்றத்தில்", "பிரச்சினைகள்", "தமிழ்த்",
    "பெருந்தோட்டக்", "சம்பள", "தோட்டத்", "பெருந்தோட்ட", "காணி",
    "கம்பனிகள்", "சம்பளம்", "மக்களை", "தொழில்", "பெருந்தோட்டத்",
    "உயர்வு", "கிட்டத்தட்ட", "சம்பள உயர்வு", "தொழிலாளர்களுக்கு",
    "பிரச்சினை", "ஆயிரம்", "தொழிலாளர்கள்", "வேலை", "தோட்ட",
    "தொழிலாளர்களின்", "சபையிலே", "இன்றைக்கு",
    "பெருந்தோட்டக் கம்பனிகள்", "பெருந்தோட்டப்", "1 000",
    "விளையாட்டு", "போக்குவரத்து", "கேட்கின்றேன்", "வீதி",
    "போக்குவரத்துச்", "புகையிரத", "பாடசாலை", "மேலதிக", "மில்லியன்",
    "நிலை", "வட", "சேவை", "பாலம்", "வீதிகள்", "காலங்களில்", "மன்னார்",
    "கிளிநொச்சி", "சேவையை", "பிரதேசத்தில்",
    "வைத்தியசாலை", "வைத்தியசாலையில்", "வைத்திய", "வைத்தியசாலைகள்",
    "வைத்தியர்கள்", "சுகாதார", "வைத்தியசாலைக்கு", "முல்லைத்தீவு",
    "ஆதார", "ஏனைய", "பொது", "எடுக்க", "வவுனியா", "உங்களுக்கு",
    "இப்போது", "மருத்துவ", "முடியுமா",
    "காணிகள்", "காணிகளை", "மீனவர்கள்", "ஏக்கர்", "இந்திய",
    "கடற்றொழில்", "விமான", "நீர்", "கேட்டுக்கொள்கின்றேன்",
    "திணைக்களம்", "அதனை", "200","இருக்கிறார்கள்","நல்ல", "அதை", "அவருடைய", "அதற்கு", "நிலைமை","தொடர்பாக", "அவர்களின்", "புதிய", "திகதி",
    "இந்தக்","இவ்வாறு","எவ்வாறு","அவர்களை","இருக்கின்றோம்","அதற்கான","எந்த","அனைத்து","அதேபோன்று","இல்லாமல்","என்றும்","இருந்தால்","முடியாத","சேர்ந்த","வேண்டிய","சம்பந்தமாக","சார்ந்த","குறித்த","நடவடிக்கைகளை","உடனடியாக","வழங்க","தவிசாளர்",
    "ஆம்","இருக்கலாம்","இடத்தில்","வேண்டுமென்று","முறையில்","மாதம்","தேவையான","உரிய","முன்னர்","இப்பொழுது","கூறி","அவை","ஒவ்வொரு","அந்தத்","இருக்க","வருகின்றது","இங்கே","அதில்","என்றால்","இதனை","இருந்தாலும்","தமது","தெரிவித்துக்","எனக்கு","ஏனென்றால்","தினம்","வருகின்ற","அண்மையில்","அன்று","நிச்சயமாக","உறுப்பினர்கள்","கெளரவ",
    "மத்தியில்", "அரசாங்கத்தின்", "குழுக்களின் பிரதித்", "காலம்", "செய்கின்ற", "நீண்ட", "எல்லோரும்", "மத்திய", "விடயத்தை", "பிரதான", "சபைத்", "தற்பொழுது", "பெரும்", "சம்பந்தமான", "அங்கே", "அவர்களிடம்", "நன்றியைத்", "வேறு", "முன்னாள்", "வாழ்கின்ற", "முக்கியமான", "பகுதிகளில்", "இல்லாத", "இணைந்து", "மக்களும்", "ரீதியான", "மேற்பட்ட", "எந்தவிதமான", "யார்", "முக்கிய", "தயாராக", "தலைவர்கள்", "அமைச்சின்", "வசதிகள்", "மூன்று", "அதனால்", "மோசமான", "கால", "விவாதத்தில்", "முழுமையாக", "உட்பட", "பற்றிய", "சம்பந்தப்பட்ட", "ம்", "முதலில்", "என்பன", "இலட்சம்", "அதற்காக", "நாட்டுக்கு"
]

# ── English stopwords ──────────────────────────────────────────────────────────
english_stops = [
    # Standard English stopwords
    "the", "and", "is", "in", "to", "of", "it", "that", "this", "for", "on", "with",
    "as", "by", "at", "be", "are", "was", "were", "from", "have", "has", "had", "not",
    "but", "or", "an", "they", "which", "you", "we", "i", "my", "me", "their", "will",
    "would", "what", "who", "whom", "these", "those", "am", "been", "being", "do",
    "does", "did", "doing", "a", "if", "because", "until", "while", "about", "against",
    "between", "into", "through", "during", "before", "after", "above", "below",
    "up", "down", "out", "off", "over", "under", "again", "further", "then", "once",
    "here", "there", "when", "where", "why", "how", "all", "any", "both", "each",
    "few", "more", "most", "other", "some", "such", "no", "nor", "only", "own",
    "same", "so", "than", "too", "very", "can", "just", "should", "now",
    "himself", "herself", "itself", "themselves", "ourselves", "yourself",
    "yourselves", "myself", "ours", "yours", "hers", "theirs", "him", "her",
    "its", "he", "she", "s",
    # Parliamentary English
    "hon", "speaker", "member", "parliament", "sir", "madam", "minister",
    "honorable", "house", "question", "answer", "debate", "bill", "act",
    "amendment", "point", "order", "chair", "time", "minute", "gentleman",
    "friend", "yes", "no", "hear", "parliamentary", "thank", "country",
    "going", "report", "like", "model", "extra", "much", "must",
    "project", "system", "come", "per", "also", "one", "mr",
    # Common high-frequency English in debates
    "us", "may", "even", "made", "say", "said", "many", "well",
    "two", "way", "go", "new", "make", "today", "could", "want", "think",
    "given", "take", "taken", "therefore", "cannot", "upon", "towards",
    "ensure", "regard", "members", "public", "state", "side", "believe",
    "high", "best", "past", "sure", "without", "bring", "discharge",
    "dignity", "elected", "hope", "experience", "career", "responsibility",
    "august", "assembly", "affairs", "wish", "mandates", "democracy",
    "congratulate", "behalf", "office", "party", "rights", "national",
    "political", "language", "honour", "election", "opposition",
    "rs", "cent", "dollars", "billion", "million", "sector", "bank",
    "global", "year", "years", "able", "provide", "development",
    "economy", "economic", "budget", "education", "tax", "debt",
    "foreign", "world", "international", "commission", "government",
    "president", "law", "court", "matter", "justice", "police",
    "committee", "case", "cases", "article", "supreme", "person",
    "prime", "leader", "issue", "north", "east", "human", "security",
    "resolution", "policy", "war", "important", "ministry", "digital",
    "north east", "human rights", "supreme court", "pta", "terrorism",
    "hospital", "medical", "hospitals", "province", "district",
    "officers", "general", "ct", "medical officers", "base",
]

# ── Merge, deduplicate, load optional file overrides ─────────────────────────
CUSTOM_STOPWORDS = sorted(set(sinhala_stops + tamil_stops + english_stops))

# Optionally merge with stopwords.json if it exists
if SW_PATH.exists():
    try:
        raw = SW_PATH.read_text(encoding='utf-8').strip()
        file_sw = list(json.loads(raw)) if raw else []
        CUSTOM_STOPWORDS = sorted(set(CUSTOM_STOPWORDS + file_sw))
        print(f'Merged {len(file_sw)} stopwords from {SW_PATH.name}')
    except Exception as e:
        print(f'Warning: Could not load {SW_PATH.name}: {e}')

print(f'Total unique stopwords: {len(CUSTOM_STOPWORDS)}')

## 9. Top Keywords per Cluster (Optimized)

In [ ]:
custom_token_pattern = r'(?u)[a-zA-Z\u0D80-\u0DFF\u0B80-\u0BFF\u200D\u200C]{2,}'

texts = df['text'].fillna('').tolist()


def top_words_optimized(labels, texts, top_n=TOP_N_WORDS, stop=CUSTOM_STOPWORDS):
    """
    Fit ONE global CountVectorizer, then extract top words per cluster
    via matrix slicing. Much faster than per-cluster fitting.
    """
    arr = np.array(labels)
    valid_mask = arr != -1
    valid_texts = [t if isinstance(t, str) else '' for t, m in zip(texts, valid_mask) if m]
    valid_labels = arr[valid_mask]

    vec = CountVectorizer(
        stop_words=stop,
        ngram_range=(1, 2),
        min_df=2,
        max_features=2000,
        token_pattern=custom_token_pattern,
    )
    X = vec.fit_transform(valid_texts)
    vocab = vec.get_feature_names_out()

    result = {}
    for cid in sorted(set(valid_labels)):
        cluster_mask = valid_labels == cid
        cluster_sums = np.asarray(X[cluster_mask].sum(axis=0)).ravel()
        top_indices  = cluster_sums.argsort()[::-1][:top_n]
        result[cid]  = [vocab[i] for i in top_indices if cluster_sums[i] > 0]

    return result


print('Extracting keywords ...')
kw_hdb = top_words_optimized(hdb_labels, texts)
kw_km  = top_words_optimized(km_labels,  texts)
kw_agg = top_words_optimized(agg_labels, texts)
print(f'Done — HDBSCAN: {len(kw_hdb)} clusters, KMeans: {len(kw_km)}, Agglom: {len(kw_agg)}')

print(f'\nHDBSCAN top keywords (first 10 clusters):')
for cid, words in list(kw_hdb.items())[:10]:
    print(f'  Cluster {cid:>3}: {", ".join(words[:10])}')

## 10. Keywords Table (all algorithms)

In [ ]:
def print_keyword_table(name, kw_dict, max_show=1000):
    sep = '=' * 62
    print(f'\n{sep}')
    print(f'  {name}  — top {TOP_N_WORDS} words per cluster')
    print(sep)
    for cid, words in list(kw_dict.items())[:max_show]:
        print(f'  {cid:>4} | {", ".join(words)}')
    extra = len(kw_dict) - max_show
    if extra > 0:
        print(f'  ... ({extra} more clusters)')


print_keyword_table('HDBSCAN',       kw_hdb)
print_keyword_table('KMeans',        kw_km)
print_keyword_table('Agglomerative', kw_agg)

In [ ]:
import json

# Combine dictionaries and force all NumPy int64 keys to become standard Python strings
master_keywords_dict = {
    "HDBSCAN": {str(k): v for k, v in kw_hdb.items()},
    "KMeans": {str(k): v for k, v in kw_km.items()},
    "Agglomerative": {str(k): v for k, v in kw_agg.items()}
}

# Define the file path
json_file_path = "cluster_keywords.json"

# Save the dictionary to a JSON file
with open(json_file_path, "w", encoding="utf-8") as f:
    json.dump(master_keywords_dict, f, ensure_ascii=False, indent=4)

print(f"✓ Keywords successfully saved to {json_file_path}")

## 11. Visualization: 2D Scatter Plots

In [ ]:
def scatter_clusters(ax, coords, labels, title, palette='tab20'):
    arr    = np.array(labels)
    unique = sorted(set(arr))
    cmap   = plt.get_cmap(palette, max(len(unique), 1))
    for i, cid in enumerate(unique):
        mask  = arr == cid
        color = '#aaaaaa' if cid == -1 else cmap(i)
        lbl   = f'noise ({mask.sum()})' if cid == -1 else f'{cid} ({mask.sum()})'
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=[color], s=3, alpha=0.5, linewidths=0, label=lbl)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('UMAP-1', fontsize=9)
    ax.set_ylabel('UMAP-2', fontsize=9)
    ax.tick_params(labelsize=7)


fig, axes = plt.subplots(1, 3, figsize=(21, 7))
scatter_clusters(axes[0], emb2, hdb_labels, f'HDBSCAN  (K={K_hdbscan})')
scatter_clusters(axes[1], emb2, km_labels,  f'KMeans   (K={K_hdbscan})')
scatter_clusters(axes[2], emb2, agg_labels, f'Agglomerative  (K={K_hdbscan})')

plt.suptitle('v11 — Full Corpus Clustering  (UMAP 2D)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scatter_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> scatter_all.png')

## 12. Visualization: Cluster Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

for ax, (name, labels) in zip(axes, [('HDBSCAN', hdb_labels),
                                       ('KMeans', km_labels),
                                       ('Agglomerative', agg_labels)]):
    arr    = np.array(labels)
    counts = pd.Series(arr).value_counts().sort_index()
    colors = ['#e74c3c' if idx == -1 else '#3498db' for idx in counts.index]
    ax.bar(range(len(counts)), counts.values, color=colors,
           edgecolor='white', linewidth=0.4)
    ax.set_title(f'{name}  (K={K_hdbscan})', fontweight='bold')
    ax.set_xlabel('Cluster ID', fontsize=9)
    ax.set_ylabel('Speech count', fontsize=9)
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels([str(i) for i in counts.index], fontsize=6, rotation=90)

plt.suptitle('Cluster Size Distribution  (red = noise)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cluster_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> cluster_sizes.png')

## 13. Visualization: Year Distribution per Cluster

In [ ]:
def year_heatmap(ax, labels, years, title):
    df_tmp = pd.DataFrame({'cluster': labels, 'year': years})
    df_tmp = df_tmp[df_tmp['cluster'] != -1]
    pivot  = df_tmp.groupby(['cluster', 'year']).size().unstack(fill_value=0)
    pct    = pivot.div(pivot.sum(axis=1), axis=0) * 100
    sns.heatmap(pct, ax=ax, cmap='YlOrRd',
                annot=(len(pct) <= 40), fmt='.0f', linewidths=0.2,
                cbar_kws={'label': '% speeches', 'shrink': 0.6})
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('Cluster', fontsize=9)
    ax.tick_params(axis='y', labelsize=7)
    ax.tick_params(axis='x', labelsize=8, rotation=45)


n_rows = max(6, K_hdbscan // 4)
fig, axes = plt.subplots(1, 3, figsize=(24, n_rows))

year_heatmap(axes[0], hdb_labels,  df['year'].tolist(), f'HDBSCAN (K={K_hdbscan})')
year_heatmap(axes[1], km_labels,   df['year'].tolist(), f'KMeans (K={K_hdbscan})')
year_heatmap(axes[2], agg_labels,  df['year'].tolist(), f'Agglomerative (K={K_hdbscan})')

plt.suptitle('Year Distribution per Cluster  (% of speeches per cluster)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'year_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> year_heatmap.png')

## 14. Temporal Topic Modeling — How Topics Evolved Over Time

In [ ]:
# ── Step 2: Temporal Topic Modeling ─────────────────────────────────────────
# For each algorithm, count speeches per (year, cluster), normalise to % of
# all speeches that year, then plot one line per topic so we can see which
# topics spiked during real-world events.

import matplotlib.ticker as mticker

ALGO_CONFIGS = [
    ('HDBSCAN',        hdb_labels,  kw_hdb),
    ('KMeans',         km_labels,   kw_km),
    ('Agglomerative',  agg_labels,  kw_agg),
]

years_arr = df['year'].values


def make_topic_label(cluster_id, kw_dict, n_words=4):
    """Short label: 'C3: word1, word2, word3'"""
    words = kw_dict.get(cluster_id, [])[:n_words]
    tag   = ', '.join(words) if words else '—'
    return f'C{cluster_id}: {tag}'


def temporal_line_chart(labels, years, kw_dict, algo_name, ax=None, top_n=None):
    """
    Plot normalised % of speeches per year for every topic.
    top_n: if set, only draw the top_n topics by total speech count
           (keeps the chart readable when K is large).
    """
    df_t = pd.DataFrame({'cluster': labels, 'year': years})
    df_t = df_t[df_t['cluster'] != -1]          # drop noise

    # raw counts: rows = cluster, cols = year
    pivot = (df_t.groupby(['cluster', 'year'])
                 .size()
                 .unstack(fill_value=0))

    # normalise: % of all speeches *in that year*
    year_totals = df_t.groupby('year').size()
    pct = pivot.div(year_totals, axis=1) * 100

    # pick topics to highlight
    if top_n is not None:
        top_clusters = pivot.sum(axis=1).nlargest(top_n).index
        pct_plot = pct.loc[top_clusters]
    else:
        pct_plot = pct

    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 6))

    cmap   = plt.get_cmap('tab20', len(pct_plot))
    x_vals = sorted(pct.columns)

    for idx, (cluster_id, row) in enumerate(pct_plot.iterrows()):
        label = make_topic_label(cluster_id, kw_dict)
        ax.plot(x_vals, [row.get(yr, 0) for yr in x_vals],
                marker='o', linewidth=1.8, markersize=4,
                color=cmap(idx), label=label)

    ax.set_title(f'{algo_name} — Topic Share Over Time',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('% of yearly speeches', fontsize=9)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1),
              fontsize=7, ncol=1, frameon=True)
    return ax


# ── How many topics to show per chart ────────────────────────────────────────
# With many clusters the legend gets crowded; cap at 20 most-active topics.
TOP_N = min(20, K_hdbscan)

fig, axes = plt.subplots(3, 1, figsize=(16, 18))

for ax, (algo_name, labels, kw_dict) in zip(axes, ALGO_CONFIGS):
    temporal_line_chart(labels, years_arr, kw_dict, algo_name,
                        ax=ax, top_n=TOP_N)

plt.suptitle(
    f'Temporal Topic Evolution  (top {TOP_N} topics by speech count)',
    fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_topics_line.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> temporal_topics_line.png')


# ── Spotlight chart: most volatile topic per algo ────────────────────────────
# "Volatile" = highest std-dev of yearly share — i.e. the topic that spiked most.
print('\n── Most volatile topic per algorithm (highest year-to-year variance) ──')

fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))

for ax, (algo_name, labels, kw_dict) in zip(axes2, ALGO_CONFIGS):
    df_t        = pd.DataFrame({'cluster': labels, 'year': years_arr})
    df_t        = df_t[df_t['cluster'] != -1]
    pivot       = df_t.groupby(['cluster', 'year']).size().unstack(fill_value=0)
    year_totals = df_t.groupby('year').size()
    pct         = pivot.div(year_totals, axis=1) * 100

    volatile_id = pct.std(axis=1).idxmax()
    label       = make_topic_label(volatile_id, kw_dict, n_words=6)
    x_vals      = sorted(pct.columns)
    y_vals      = [pct.loc[volatile_id].get(yr, 0) for yr in x_vals]

    ax.fill_between(x_vals, y_vals, alpha=0.25, color='steelblue')
    ax.plot(x_vals, y_vals, marker='o', linewidth=2.2,
            color='steelblue', label=label)
    ax.set_title(f'{algo_name}\n{label}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Year', fontsize=8)
    ax.set_ylabel('% of yearly speeches', fontsize=8)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    print(f'  {algo_name}: {label}')

plt.suptitle('Most Volatile Topic per Algorithm',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_topics_spotlight.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> temporal_topics_spotlight.png')


## 14. Save Results

In [ ]:
df_out = df[['speech_id', 'date', 'year', 'speaker']].copy()
df_out['hdbscan_cluster']       = hdb_labels
df_out['kmeans_cluster']        = km_labels
df_out['agglomerative_cluster'] = agg_labels

out_csv = OUTPUT_DIR / 'v11_cluster_assignments.csv'
df_out.to_csv(out_csv, index=False)
print(f'Saved -> {out_csv}')

df_metrics.to_csv(OUTPUT_DIR / 'v11_metrics.csv')
print(f'Saved -> {OUTPUT_DIR / "v11_metrics.csv"}')

kw_path = OUTPUT_DIR / 'v11_top_keywords.json'
with open(kw_path, 'w', encoding='utf-8') as f:
    json.dump(
        {'hdbscan'       : {str(k): v for k, v in kw_hdb.items()},
         'kmeans'        : {str(k): v for k, v in kw_km.items()},
         'agglomerative' : {str(k): v for k, v in kw_agg.items()}},
        f, indent=2, ensure_ascii=False)
print(f'Saved -> {kw_path}')
print('\nAll done ✓')

In [ ]:
# ── Cell: Generate LLM Macro-Categorization Prompt ──────────────────────────
llm_prompt_path = OUTPUT_DIR / "llm_macro_topic_prompt.txt"

with open(llm_prompt_path, "w", encoding="utf-8") as f:
    # 1. Write the instructions for Claude
    f.write("I am providing a list of micro-topics extracted from Sri Lankan parliamentary debates, represented by their top keywords. I cannot analyze these topics individually. Please act as a Political Data Scientist. Read these micro-topics and group them into exactly 10 to 15 overarching 'Macro-Categories' (e.g., Economy & IMF, Healthcare & Covid, National Security, Infrastructure, Agriculture, etc.).\n\n")
    
    # 2. Crucial: Force Claude to write Python code for you!
    f.write("Output ONLY a clean Python dictionary where the keys are the Macro-Category names (strings) and the values are lists of integers (the Micro-Topic IDs). Do not include any other explanations.\n\n")
    
    f.write("--- MICRO TOPICS ---\n")
    
    # 3. Write the topics (skipping the -1 noise cluster)
    for cluster_id, words in kw_hdb.items():
        if cluster_id == -1:
            continue
        
        # Take just the top 10 words for the LLM
        top_10 = ", ".join(words[:10])
        f.write(f"Topic {cluster_id}: {top_10}\n")

print(f"✓ LLM Prompt generated and saved to: {llm_prompt_path}")
print("Open this file, copy ALL the text, and paste it into Claude!")

In [ ]:
# from scipy.cluster.hierarchy import linkage, fcluster
# from scipy.spatial.distance import pdist

# dist_vec = pdist(valid_centroids, metric='cosine')
# Z = linkage(dist_vec, method='average')

# n = len(valid_micro_ids)
# MIN_K, MAX_K = 29, 50  # thesis-sensible range

# # Merges are ordered: Z[n-k] is the merge that produces k clusters
# # Search only within that window
# search = Z[n - MAX_K : n - MIN_K + 1, 2]
# gaps   = np.diff(search)
# best_idx = (n - MAX_K) + np.argmax(gaps)
# best_cut = Z[best_idx, 2] + 1e-6

# labels = fcluster(Z, t=best_cut, criterion='distance')
# N_MACRO = labels.max()
# print(f"Data-driven N_MACRO = {N_MACRO}  (cut at d={best_cut:.4f})")

In [ ]:
# ============================================================================
# MACRO-TOPIC AGGREGATION v2 — Probability-Weighted Centroids + KNN Rescue
# ============================================================================

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_distances

# ── Tuneable parameters ──────────────────────────────────────────────────────
MIN_SPEECHES        = 50     # micro-topics below this → attempt KNN rescue
N_MACRO             = 30     # final macro-topic count
NICHE_RESCUE_THRESH = 0.25   # max cosine distance to absorb a niche topic
                             # (0 = identical, 1 = orthogonal; tune 0.2–0.4)
VARIANCE_WARN_THRESH = 0.15  # warn if a macro-topic's mean intra-distance > this

# ── Attach labels & probabilities to df ─────────────────────────────────────
# hdb is the fitted HDBSCAN object; hdb_labels and hdb.probabilities_ align
# with df's integer index.
df['hdbscan_cluster']     = hdb_labels
df['hdbscan_probability'] = hdb.probabilities_   # confidence score ∈ [0, 1]
umap_embeddings           = emb5


# ── Step 1: Size-Based Filtering ─────────────────────────────────────────────
cluster_counts  = (
    df[df['hdbscan_cluster'] != -1]['hdbscan_cluster']
    .value_counts()
)

valid_micro_ids = cluster_counts[cluster_counts >= MIN_SPEECHES].index.tolist()
niche_micro_ids = cluster_counts[cluster_counts <  MIN_SPEECHES].index.tolist()

print(f"Total micro-topics (excl. noise) : {len(cluster_counts)}")
print(f"Valid  (≥{MIN_SPEECHES} speeches): {len(valid_micro_ids)}")
print(f"Niche  (<{MIN_SPEECHES} speeches): {len(niche_micro_ids)}")
print(f"Noise  (-1)                      : {(df['hdbscan_cluster'] == -1).sum()}")

assert len(valid_micro_ids) >= N_MACRO, (
    f"Only {len(valid_micro_ids)} valid micro-topics but N_MACRO={N_MACRO}. "
    f"Lower MIN_SPEECHES or reduce N_MACRO."
)


# ── Step 2: Probability-Weighted Centroid Calculation ────────────────────────
# np.average(..., weights=probs) gives high-confidence core points more pull
# than uncertain border points, producing a cleaner centroid.
# Edge case: if all probabilities in a cluster are 0 (rare), fall back to mean.

valid_centroids = []

for micro_id in valid_micro_ids:
    indices = np.where((df['hdbscan_cluster'] == micro_id).values)[0]
    probs   = df.iloc[indices]['hdbscan_probability'].values

    if probs.sum() == 0:                          # degenerate fallback
        centroid = umap_embeddings[indices].mean(axis=0)
    else:
        centroid = np.average(umap_embeddings[indices], axis=0, weights=probs)

    valid_centroids.append(centroid)

valid_centroids = np.array(valid_centroids)       # (n_valid, embedding_dim)
print(f"\nValid centroid matrix : {valid_centroids.shape}")


# ── Step 3: Hierarchical Merging on Valid Centroids ──────────────────────────
agg = AgglomerativeClustering(
    n_clusters = N_MACRO,
    metric     = 'cosine',
    linkage    = 'average',
)
macro_assignments = agg.fit_predict(valid_centroids)
# macro_assignments[i] ∈ {0..N_MACRO-1} for valid_micro_ids[i]


# ── Step 3b: Variance / Cohesion Check ───────────────────────────────────────
# For each macro-topic, compute the mean pairwise cosine distance among its
# constituent micro-topic centroids. High variance = forced merge of
# semantically distant themes — worth flagging before you name the topic.

print("\n── Macro-Topic Cohesion Check ───────────────────────────────────────")
macro_centroid_map = {}   # macro_id → mean centroid of its micro-centroids

for macro_id in range(N_MACRO):
    member_mask     = macro_assignments == macro_id
    member_centroids = valid_centroids[member_mask]
    macro_centroid_map[macro_id] = member_centroids.mean(axis=0)

    if len(member_centroids) > 1:
        dists        = cosine_distances(member_centroids)
        mean_dist    = dists[np.triu_indices_from(dists, k=1)].mean()
        flag         = " ⚠ HIGH VARIANCE" if mean_dist > VARIANCE_WARN_THRESH else ""
        n_micros     = member_mask.sum()
        print(f"  Macro-{macro_id:02d}: {n_micros} micro-topics, "
              f"mean cosine dist = {mean_dist:.3f}{flag}")
    else:
        print(f"  Macro-{macro_id:02d}: 1 micro-topic (singleton)")


# ── Step 4: Build macro centroid array for KNN rescue ────────────────────────
macro_centroids = np.array([macro_centroid_map[i] for i in range(N_MACRO)])


# ── Step 5: KNN Rescue — absorb niche topics into nearest macro-topic ────────
# Instead of dumping all small clusters into one noisy bucket, each niche
# micro-topic gets a cosine distance check against the 30 macro centroids.
# If close enough → absorbed. Otherwise → stays "Niche/Localized Debates".

niche_centroids = []
for micro_id in niche_micro_ids:
    indices = np.where((df['hdbscan_cluster'] == micro_id).values)[0]
    probs   = df.iloc[indices]['hdbscan_probability'].values
    if probs.sum() == 0:
        c = umap_embeddings[indices].mean(axis=0)
    else:
        c = np.average(umap_embeddings[indices], axis=0, weights=probs)
    niche_centroids.append(c)

rescued, left_as_niche = 0, 0
niche_rescue_map = {}     # niche micro_id → macro label

if niche_centroids:
    niche_centroids = np.array(niche_centroids)
    dist_matrix = cosine_distances(niche_centroids, macro_centroids)
    # dist_matrix[i, j] = cosine distance between niche i and macro j

    for i, micro_id in enumerate(niche_micro_ids):
        nearest_macro = dist_matrix[i].argmin()
        nearest_dist  = dist_matrix[i, nearest_macro]

        if nearest_dist <= NICHE_RESCUE_THRESH:
            niche_rescue_map[micro_id] = f"Macro-Topic {nearest_macro}"
            rescued += 1
        else:
            niche_rescue_map[micro_id] = "Niche/Localized Debates"
            left_as_niche += 1

print(f"\nNiche rescue: {rescued} absorbed into macro-topics, "
      f"{left_as_niche} remain as Niche/Localized Debates")


# ── Step 6: Mapping Back to the Dataframe ────────────────────────────────────
micro_to_macro_dict = {}

# Valid micro-topics → their assigned macro-topic
for micro_id, macro_id in zip(valid_micro_ids, macro_assignments):
    micro_to_macro_dict[micro_id] = f"Macro-Topic {macro_id}"

# Niche micro-topics → rescued or left as niche
micro_to_macro_dict.update(niche_rescue_map)

# HDBSCAN noise → stays as noise
micro_to_macro_dict[-1] = "Procedural Noise"

df['macro_topic'] = df['hdbscan_cluster'].map(micro_to_macro_dict)

unmapped = df['macro_topic'].isna().sum()
if unmapped > 0:
    print(f"\nWARNING: {unmapped} rows unmapped — check hdbscan_cluster values.")
else:
    print("All rows mapped successfully ✓")


# ── Step 7: Verification ─────────────────────────────────────────────────────
print("\n── Final Macro-Topic Distribution ──────────────────────────────────")
print(df['macro_topic'].value_counts().to_string())
print(f"\nDistinct macro_topic values: {df['macro_topic'].nunique()}")


In [ ]:
# ── Save speech-level macro_topic mapping ─────────────────────────────────────
macro_map_path = OUTPUT_DIR / "macro_topic_assignments.csv"
df[["speech_id", "macro_topic"]].to_csv(macro_map_path, index=False)
print(f"Saved macro_topic assignments -> {macro_map_path}")
print(df["macro_topic"].value_counts().to_string())

In [ ]:
"""
Publication-quality UMAP scatter — Macro-Topic Assignments.
- Clean white background, no KDE wash
- Larger dots filling the full plot area
- MT-id centroid labels only, no legend
- Matches aesthetic of bitopic_paper_figure.py
"""

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from matplotlib import rcParams

rcParams['font.family'] = 'serif'
rcParams['font.serif']  = ['Times New Roman', 'DejaVu Serif', 'serif']

# ── Config ────────────────────────────────────────────────────────────────────
FIG_WIDTH   = 7.0
FIG_HEIGHT  = 5.5
DPI_SCREEN  = 130
DPI_SAVE    = 300

POINT_SIZE   = 6
POINT_ALPHA  = 0.75
NOISE_SIZE   = 3
NOISE_ALPHA  = 0.12
EDGE_WIDTH   = 0.20        # thin black outline like the bitopic figure
LABEL_FS     = 6.5
TITLE_FS     = 9
AXIS_FS      = 7.5

# ── Palette ───────────────────────────────────────────────────────────────────
PALETTE_30 = [
    '#E63946','#F4A261','#2A9D8F','#457B9D','#6A0572',
    '#F4D03F','#264653','#E76F51','#90BE6D','#43AA8B',
    '#577590','#C77DFF','#FF6B6B','#48CAE4','#F77F00',
    '#06D6A0','#118AB2','#EF476F','#FFD166','#83C5BE',
    '#BC6C25','#606C38','#DDA15E','#A8DADC','#C1121F',
    '#669BBC','#FFC8DD','#B5838D','#6D6875','#E9C46A',
]

unique_macros = sorted(df['macro_topic'].unique(),
                       key=lambda m: (('Noise' in str(m) or 'Niche' in str(m)), str(m)))
color_map = {}
palette_idx = 0
for m in unique_macros:
    if 'Noise' in str(m) or 'Niche' in str(m):
        color_map[m] = '#CCCCCC'
    else:
        color_map[m] = PALETTE_30[palette_idx % len(PALETTE_30)]
        palette_idx += 1

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT), dpi=DPI_SCREEN)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

x_all = emb2[:, 0]
y_all = emb2[:, 1]
pad   = 0.02
XLIM  = (np.percentile(x_all, 0.05)  - pad, np.percentile(x_all, 99.95) + pad)
YLIM  = (np.percentile(y_all, 0.05)  - pad, np.percentile(y_all, 99.95) + pad)

# Scale FIG_HEIGHT to match the actual data aspect ratio so no blank space
data_ratio = (YLIM[1] - YLIM[0]) / (XLIM[1] - XLIM[0])
FIG_HEIGHT  = FIG_WIDTH * data_ratio
fig.set_size_inches(FIG_WIDTH, FIG_HEIGHT)

ax.set_xlim(XLIM)
ax.set_ylim(YLIM)


# ── Scatter: noise first (background), then clusters on top ──────────────────
for m, grp in df.groupby('macro_topic'):
    if not ('Noise' in str(m) or 'Niche' in str(m)):
        continue
    pts = emb2[grp.index.tolist()]
    ax.scatter(pts[:, 0], pts[:, 1],
               c='#CCCCCC', s=NOISE_SIZE, alpha=NOISE_ALPHA,
               linewidths=0, zorder=1, rasterized=True)

for m, grp in df.groupby('macro_topic'):
    if 'Noise' in str(m) or 'Niche' in str(m):
        continue
    pts = emb2[grp.index.tolist()]
    ax.scatter(pts[:, 0], pts[:, 1],
               c=color_map[m],
               s=POINT_SIZE, alpha=POINT_ALPHA,
               edgecolors='#1a1a1a', linewidths=EDGE_WIDTH,
               zorder=2, rasterized=True)

# ── Centroid labels ───────────────────────────────────────────────────────────
for macro_id, centroid_vec in macro_centroid_map.items():
    dists_2d  = np.linalg.norm(emb5 - centroid_vec, axis=1)
    proxy_idx = np.argmin(dists_2d)
    cx, cy    = emb2[proxy_idx]
    ax.text(cx, cy, f"MT-{macro_id}",
            fontsize=LABEL_FS, fontweight='bold',
            ha='center', va='center', zorder=5,
            color='#111111',
            bbox=dict(boxstyle='round,pad=0.18',
                      fc='white', ec='none', alpha=0.80))

# ── Axes styling ──────────────────────────────────────────────────────────────
ax.set_xlabel('UMAP-1', fontsize=AXIS_FS, labelpad=3)
ax.set_ylabel('UMAP-2', fontsize=AXIS_FS, labelpad=3)
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_linewidth(0.5)
    spine.set_color('#AAAAAA')

ax.set_title('UMAP 2-D Projection — Macro-Topic Assignments',
             fontsize=TITLE_FS, fontweight='bold', pad=6)

# ── Save ──────────────────────────────────────────────────────────────────────
plt.tight_layout(pad=0.8)

png_path = OUTPUT_DIR / 'macro_topic_umap_scatter.png'
pdf_path = OUTPUT_DIR / 'macro_topic_umap_scatter.pdf'
plt.savefig(png_path, dpi=DPI_SAVE, bbox_inches='tight', facecolor='white')
plt.savefig(pdf_path, dpi=DPI_SAVE, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved PNG: {png_path}")
print(f"Saved PDF: {pdf_path}")

In [ ]:
# ============================================================================
# MACRO-TOPIC KEYWORD EXTRACTION
# CountVectorizer  → top frequency words per macro-topic
# TF-IDF           → top distinctive words per macro-topic
# ============================================================================

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

TOP_N_MACRO_WORDS = 100    # keywords to extract per macro-topic

# ── Reuse the same token pattern & stopwords from cell 20 ────────────────────
macro_texts  = df['text'].fillna('').tolist()
macro_labels = df['macro_topic'].tolist()

# Only operate on actual macro-topics (exclude Noise / Niche buckets)
valid_mask   = ~df['macro_topic'].isin(['Procedural Noise', 'Niche/Localized Debates'])
filtered_texts  = [t for t, v in zip(macro_texts,  valid_mask) if v]
filtered_labels = [l for l, v in zip(macro_labels, valid_mask) if v]

print(f"Speeches going into keyword extraction: {len(filtered_texts)}")
print(f"Macro-topics to label: {len(set(filtered_labels))}\n")


# ── A: CountVectorizer — top raw-frequency words ─────────────────────────────
# Same as your existing top_words_optimized but operating on macro_topic labels.
# Good for: understanding the dominant vocabulary of each macro-topic.

cv = CountVectorizer(
    stop_words    = CUSTOM_STOPWORDS,
    ngram_range   = (1, 2),
    min_df        = 3,
    max_features  = 5000,
    token_pattern = custom_token_pattern,
)
X_count = cv.fit_transform(filtered_texts)
cv_vocab = cv.get_feature_names_out()

kw_macro_count = {}
for macro_id in sorted(set(filtered_labels)):
    mask         = np.array(filtered_labels) == macro_id
    sums         = np.asarray(X_count[mask].sum(axis=0)).ravel()
    top_idx      = sums.argsort()[::-1][:TOP_N_MACRO_WORDS]
    kw_macro_count[macro_id] = [(cv_vocab[i], int(sums[i])) for i in top_idx if sums[i] > 0]


# ── B: TF-IDF — top distinctive words ────────────────────────────────────────
# TF-IDF down-weights words that appear across ALL macro-topics (e.g. "said",
# "parliament") and surfaces words that are *unique* to each macro-topic.
# Good for: naming the topic and thesis narrative.

tfidf = TfidfVectorizer(
    stop_words    = CUSTOM_STOPWORDS,
    ngram_range   = (1, 2),
    min_df        = 3,
    max_features  = 5000,
    token_pattern = custom_token_pattern,
    sublinear_tf  = True,    # log(1+tf) — dampens effect of very frequent words
)

# Treat each macro-topic as one "document" by joining its speeches
macro_docs   = {}
for label, text in zip(filtered_labels, filtered_texts):
    macro_docs.setdefault(label, []).append(text)

macro_doc_ids   = sorted(macro_docs.keys())
macro_doc_texts = [' '.join(macro_docs[mid]) for mid in macro_doc_ids]

X_tfidf   = tfidf.fit_transform(macro_doc_texts)
tf_vocab  = tfidf.get_feature_names_out()

kw_macro_tfidf = {}
for i, macro_id in enumerate(macro_doc_ids):
    row     = np.asarray(X_tfidf[i].todense()).ravel()
    top_idx = row.argsort()[::-1][:TOP_N_MACRO_WORDS]
    kw_macro_tfidf[macro_id] = [(tf_vocab[j], round(float(row[j]), 4))
                                 for j in top_idx if row[j] > 0]


# ── Print results side-by-side ────────────────────────────────────────────────
sep = '─' * 70
print(sep)
print(f"{'MACRO-TOPIC':<22}  {'COUNT (freq)':^30}  {'TF-IDF (distinctive)':^30}")
print(sep)

for macro_id in macro_doc_ids:
    freq_words  = ', '.join(w for w, _ in kw_macro_count.get(macro_id, [])[:16])
    tfidf_words = ', '.join(w for w, _ in kw_macro_tfidf.get(macro_id, [])[:16])
    print(f"{str(macro_id):<22}  {freq_words:<30}  {tfidf_words}")
    print()

print(sep)


# ── Save keyword dicts for downstream use ────────────────────────────────────
import json as _json

kw_out = {
    'count' : {str(k): [w for w, _ in v] for k, v in kw_macro_count.items()},
    'tfidf' : {str(k): [w for w, _ in v] for k, v in kw_macro_tfidf.items()},
    'count_with_freq': {str(k): [{'keyword': w, 'count': c} for w, c in v] for k, v in kw_macro_count.items()},
}
kw_path = OUTPUT_DIR / 'macro_topic_keywords_100.json'
with open(kw_path, 'w', encoding='utf-8') as f:
    _json.dump(kw_out, f, ensure_ascii=False, indent=2)


## 15. Topic Coherence Scores (c_v / NPMI)

Coherence measures whether the top keywords of a topic co-occur in actual speeches — a linguistic quality metric independent of cluster geometry. c_v correlates best with human judgement; NPMI is the stricter standard used in NLP papers.

In [ ]:
# # ── Topic Coherence: c_v and NPMI for HDBSCAN macro-topics ──────────────────
# # Uses Gensim's CoherenceModel. We evaluate at the macro-topic level because
# # macro-topics are the final deliverable; micro-topic coherence is also computed
# # for completeness.

# from gensim.corpora import Dictionary
# from gensim.models.coherencemodel import CoherenceModel

# # ── Tokenise corpus (reuse same token pattern) ───────────────────────────────
# import re
# _tok = re.compile(custom_token_pattern)

# def tokenize(text):
#     return _tok.findall(str(text).lower()) if isinstance(text, str) else []

# print('Tokenising corpus ...')
# tokenized_corpus = [tokenize(t) for t in texts]
# dictionary = Dictionary(tokenized_corpus)
# print(f'  Vocabulary size: {len(dictionary):,} tokens')

# # ── Helper ────────────────────────────────────────────────────────────────────
# def coherence_scores(topic_keyword_lists, label=''):
#     """Compute c_v and NPMI for a list of keyword lists."""
#     # Filter to topics that have keywords
#     valid = [kws for kws in topic_keyword_lists if kws]

#     cv_model = CoherenceModel(
#         topics=valid, texts=tokenized_corpus,
#         dictionary=dictionary, coherence='c_v'
#     )
#     npmi_model = CoherenceModel(
#         topics=valid, texts=tokenized_corpus,
#         dictionary=dictionary, coherence='c_npmi'
#     )
#     cv   = cv_model.get_coherence()
#     npmi = npmi_model.get_coherence()
#     print(f'  {label:<30}  c_v={cv:.4f}   NPMI={npmi:.4f}')
#     return cv, npmi


# print('\nComputing coherence scores ...')

# # Macro-topic keywords (TF-IDF distinctive words — best for coherence)
# macro_tfidf_lists = [
#     [w for w, _ in kw_macro_tfidf.get(mid, [])][:10]
#     for mid in sorted(kw_macro_tfidf)
#     if mid not in ('Procedural Noise', 'Niche/Localized Debates')
# ]

# # Micro-topic keywords (HDBSCAN, top 10 per cluster)
# micro_hdb_lists = [kw_hdb.get(cid, [])[:10] for cid in sorted(kw_hdb) if cid != -1]

# cv_macro, npmi_macro = coherence_scores(macro_tfidf_lists,    'Macro-Topics (30, TF-IDF)')
# cv_micro, npmi_micro = coherence_scores(micro_hdb_lists[:50], 'Micro-Topics (HDBSCAN top-50)')

# # ── Summary table ─────────────────────────────────────────────────────────────
# import pandas as pd
# coh_df = pd.DataFrame([
#     {'Level': 'Macro-Topics (30)', 'Method': 'HDBSCAN + Agglomerative', 'c_v': round(cv_macro, 4), 'NPMI': round(npmi_macro, 4)},
#     {'Level': 'Micro-Topics',      'Method': 'HDBSCAN (top 50)',        'c_v': round(cv_micro, 4), 'NPMI': round(npmi_micro, 4)},
# ])
# print()
# print(coh_df.to_string(index=False))
# print()
# print('Interpretation:')
# print('  c_v > 0.5  = acceptable   |  c_v > 0.6 = good   |  c_v > 0.7 = excellent')
# print('  NPMI > 0.1 = acceptable   |  NPMI > 0.2 = good')


## 16. Speaker / Political Actor Analysis

Which MPs dominate which macro-topics? This is the most politically interesting finding of the thesis.

In [ ]:
# ── 16.1  Speaker × Macro-Topic speech count matrix ─────────────────────────
# Only count speeches assigned to actual macro-topics (exclude Noise/Niche).
df_named = df[~df['macro_topic'].isin(['Procedural Noise', 'Niche/Localized Debates'])].copy()

speaker_topic = (
    df_named.groupby(['speaker', 'macro_topic'])
    .size()
    .unstack(fill_value=0)
)

# Keep top N speakers by total speech count across all macro-topics
TOP_SPEAKERS = 30
top_speakers = speaker_topic.sum(axis=1).nlargest(TOP_SPEAKERS).index
speaker_topic_top = speaker_topic.loc[top_speakers]

# Row-normalise so each row shows % of that speaker's speeches per macro-topic
# This prevents prolific speakers from dominating purely by volume.
speaker_pct = speaker_topic_top.div(speaker_topic_top.sum(axis=1), axis=0) * 100

print(f'Speaker-topic matrix: {speaker_topic.shape}  (all speakers)')
print(f'Showing top {TOP_SPEAKERS} speakers by speech count')

# ── 16.2  Heatmap ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    speaker_pct,
    cmap       = 'YlOrRd',
    annot      = False,
    linewidths = 0.2,
    ax         = ax,
    cbar_kws   = {'label': '% of speaker\'s speeches', 'shrink': 0.6},
)
ax.set_title(f'Top {TOP_SPEAKERS} Speakers × Macro-Topic  (% of each speaker\'s speeches)',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Macro-Topic', fontsize=9)
ax.set_ylabel('Speaker', fontsize=9)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'speaker_topic_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> speaker_topic_heatmap.png')

# ── 16.3  Top 5 speakers per macro-topic table ───────────────────────────────
print('\n── Top 5 Speakers per Macro-Topic (by speech count) ────────────────────')
for macro_id in sorted(speaker_topic.columns):
    top5 = speaker_topic[macro_id].nlargest(5)
    names = ', '.join(f'{s} ({n})' for s, n in top5.items() if n > 0)
    print(f'  {str(macro_id):<28}: {names}')


## 17. Language-Stratified Topic Analysis

Proves macro-topics are semantically coherent, not language-contaminated. A topic dominated by one language script is not a real thematic cluster — it's a language cluster.

In [ ]:
# ── Detect dominant script per speech using Unicode block counts ──────────────
# Sinhala: U+0D80–U+0DFF  |  Tamil: U+0B80–U+0BFF  |  Latin (English): A-Z

import unicodedata

def dominant_language(text):
    """Return the dominant script: 'Sinhala', 'Tamil', or 'English'."""
    if not isinstance(text, str) or len(text) == 0:
        return 'Unknown'
    si = sum(1 for c in text if '\u0D80' <= c <= '\u0DFF')
    ta = sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')
    en = sum(1 for c in text if c.isascii() and c.isalpha())
    counts = {'Sinhala': si, 'Tamil': ta, 'English': en}
    return max(counts, key=counts.get)

print('Detecting language per speech ...')
df['language'] = df['text'].map(dominant_language)
lang_dist = df['language'].value_counts()
print(lang_dist.to_string())

# ── Per-macro-topic language breakdown ────────────────────────────────────────
df_named2 = df[~df['macro_topic'].isin(['Procedural Noise', 'Niche/Localized Debates'])].copy()

lang_topic = (
    df_named2.groupby(['macro_topic', 'language'])
    .size()
    .unstack(fill_value=0)
)

# Normalise to % per topic
lang_topic_pct = lang_topic.div(lang_topic.sum(axis=1), axis=0) * 100

# ── Stacked bar chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
lang_topic_pct.plot(
    kind   = 'barh',
    stacked= True,
    ax     = ax,
    color  = {'Sinhala': '#2196F3', 'Tamil': '#FF9800', 'English': '#4CAF50',
               'Unknown': '#9E9E9E'},
    edgecolor='white',
    width  = 0.7,
)
ax.set_title('Language Distribution per Macro-Topic\n'
             '(uniform mix = topic not language-contaminated)',
             fontweight='bold', fontsize=11)
ax.set_xlabel('% of speeches', fontsize=9)
ax.set_ylabel('Macro-Topic', fontsize=9)
ax.legend(title='Language', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.axvline(33.3, color='grey', linestyle=':', alpha=0.5, label='Equal split')
ax.axvline(66.6, color='grey', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'language_per_macro_topic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> language_per_macro_topic.png')

# ── Flag potentially contaminated topics ─────────────────────────────────────
print('\n── Language Contamination Check (topics where one language > 80%) ───────')
for topic, row in lang_topic_pct.iterrows():
    dominant_lang = row.idxmax()
    dominant_pct  = row.max()
    flag = ' ⚠ POSSIBLE LANGUAGE BIAS' if dominant_pct > 80 else ''
    print(f'  {str(topic):<28} {dominant_lang:<10} {dominant_pct:.1f}%{flag}')


## 18. α/β Ablation Study (BiTopic Fusion Weights)

Tests a grid of α (semantic weight) and β (lexical weight) values on the pre-computed UMAP projections from v13. Fast because UMAP is not re-run — only HDBSCAN is re-fit on the existing 5-D projections.

In [ ]:
# ── α/β Ablation on late-fusion UMAP projections from v13 ────────────────────
# Loads bert_emb5 and lex_emb5 (both already computed and cached in v13).
# Re-runs HDBSCAN with different fusion weights — no UMAP recompute needed.

from sklearn.preprocessing import normalize as sk_normalize
from hdbscan import HDBSCAN as _HDBSCAN

V13_DIR = OUTPUT_DIR.parent / 'v13_bitopic'

bert_emb5_path = V13_DIR / 'bert_umap5.npy'
lex_emb5_path  = V13_DIR / 'latefusion_lex_umap5.npy'

if not bert_emb5_path.exists() or not lex_emb5_path.exists():
    print('v13 UMAP files not found — run v13 notebook first.')
else:
    print('Loading v13 UMAP projections ...')
    bert5 = np.load(bert_emb5_path)
    lex5  = np.load(lex_emb5_path)
    assert len(bert5) == len(lex5), 'Shape mismatch between UMAP files'
    print(f'  bert5: {bert5.shape},  lex5: {lex5.shape}')

    # ── Grid definition ───────────────────────────────────────────────────────
    alphas = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 1.00]
    # β is always 1 - α

    results = []
    for alpha in alphas:
        beta  = round(1.0 - alpha, 2)
        sem   = sk_normalize(bert5, norm='l2')
        fused = alpha * sem
        if beta > 0:
            lex   = sk_normalize(lex5, norm='l2')
            fused = np.hstack([alpha * sem, beta * lex])

        hdb_ab = _HDBSCAN(
            min_cluster_size=MIN_CLUSTER_SIZE,
            min_samples=MIN_SAMPLES,
            metric='euclidean',
            cluster_selection_method='eom',
        )
        labels_ab = hdb_ab.fit_predict(fused)

        K_ab     = len(set(labels_ab)) - (1 if -1 in labels_ab else 0)
        noise_ab = int((labels_ab == -1).sum())
        results.append({
            'α (semantic)': alpha,
            'β (lexical)' : beta,
            'Topics (K)'  : K_ab,
            'Noise count' : noise_ab,
            'Noise %'     : round(100 * noise_ab / len(labels_ab), 2),
            'Clustered %' : round(100 * (len(labels_ab) - noise_ab) / len(labels_ab), 2),
        })
        print(f'  α={alpha:.2f} β={beta:.2f}  K={K_ab:>4}  noise={noise_ab:>5} ({results[-1]["Noise %"]:.1f}%)')

    ablation_df = pd.DataFrame(results)
    print()
    print(ablation_df.to_string(index=False))

    # ── Visualisation: dual y-axis line chart ─────────────────────────────────
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    ax1.plot(ablation_df['α (semantic)'], ablation_df['Noise %'],
             marker='o', color='crimson', linewidth=2, label='Noise %')
    ax2.plot(ablation_df['α (semantic)'], ablation_df['Topics (K)'],
             marker='s', color='steelblue', linewidth=2, linestyle='--', label='Topics (K)')

    # Mark chosen value
    chosen_alpha = 0.85
    ax1.axvline(chosen_alpha, color='grey', linestyle=':', alpha=0.7, label=f'Chosen α={chosen_alpha}')

    ax1.set_xlabel('α  (semantic weight;  β = 1 – α)', fontsize=10)
    ax1.set_ylabel('Noise %', color='crimson', fontsize=10)
    ax2.set_ylabel('Number of Topics (K)', color='steelblue', fontsize=10)
    ax1.tick_params(axis='y', colors='crimson')
    ax2.tick_params(axis='y', colors='steelblue')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

    plt.title('α/β Ablation Study — Effect of Fusion Weights on Clustering Quality',
              fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ablation_alpha_beta.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved -> ablation_alpha_beta.png')

    ablation_df.to_csv(OUTPUT_DIR / 'ablation_results.csv', index=False)
    print('Saved -> ablation_results.csv')


## 19. Word Clouds per Macro-Topic

Required qualitative visualization for the Results chapter.

In [ ]:
"""
Word clouds per macro-topic.
Color rule: words colored by TF-IDF rank using a continuous colormap
(highest score = one end, lowest = other end).
A shared colorbar on the right shows the High→Low frequency scale.
"""

from wordcloud import WordCloud
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.colorbar as mcolorbar
from matplotlib.colors import Normalize
import math
import numpy as np
from matplotlib import rcParams

rcParams['font.family'] = 'serif'
rcParams['font.serif']  = ['Times New Roman', 'DejaVu Serif', 'serif']

# ── Colormap: vivid gradient across all words ─────────────────────────────────
# 'turbo' goes deep-blue → cyan → green → yellow → orange → red
# Highest rank (most important) = red end, lowest = blue end
CMAP = cm.get_cmap('turbo')   # swap to 'plasma', 'RdYlGn', 'Spectral' etc.

macro_ids_clean = [
    mid for mid in sorted(kw_macro_tfidf)
    if mid not in ('Procedural Noise', 'Niche/Localized Debates')
]

def make_colormap_func(word_freq):
    """Map each word's rank → a colour from CMAP. Rank 0 = highest = red end."""
    ranked   = sorted(word_freq, key=word_freq.get, reverse=True)
    rank_map = {w: i for i, w in enumerate(ranked)}
    n        = max(len(ranked) - 1, 1)

    def color_func(word, font_size, position, orientation,
                   random_state=None, **kwargs):
        rank  = rank_map.get(word, n)
        # 0 = top of cmap (red/bright), 1 = bottom (blue/dark)
        t     = rank / n
        rgba  = CMAP(1.0 - t)          # invert so highest = warm end
        return f"rgb({int(rgba[0]*255)},{int(rgba[1]*255)},{int(rgba[2]*255)})"
    return color_func

# ── Layout: leave room on the right for the colorbar ─────────────────────────
n_topics = len(macro_ids_clean)
n_cols   = 5
n_rows   = math.ceil(n_topics / n_cols)

fig = plt.figure(figsize=(n_cols * 4 + 1.2, n_rows * 3), dpi=130)
fig.patch.set_facecolor('white')

# Word-cloud axes grid (leave 5% on right for colorbar)
gs = fig.add_gridspec(n_rows, n_cols,
                      left=0.01, right=0.88,
                      top=0.94, bottom=0.02,
                      hspace=0.35, wspace=0.05)

axes_flat = [fig.add_subplot(gs[r, c])
             for r in range(n_rows) for c in range(n_cols)]

for idx, macro_id in enumerate(macro_ids_clean):
    ax        = axes_flat[idx]
    word_freq = {w: score for w, score in kw_macro_tfidf.get(macro_id, [])}

    if word_freq:
        wc = WordCloud(
            width             = 400,
            height            = 250,
            background_color  = 'white',
            color_func        = make_colormap_func(word_freq),
            max_words         = 30,
            prefer_horizontal = 0.9,
            font_path         = r'C:\Windows\Fonts\Nirmala.ttc',
        ).generate_from_frequencies(word_freq)
        ax.imshow(wc, interpolation='bilinear')
    else:
        ax.text(0.5, 0.5, 'No keywords', ha='center', va='center',
                transform=ax.transAxes, fontsize=9, color='grey')

    ax.set_title(str(macro_id), fontsize=8, fontweight='bold', pad=3)
    ax.axis('off')

for ax in axes_flat[n_topics:]:
    ax.axis('off')

# ── Colorbar on the right ─────────────────────────────────────────────────────
cbar_ax = fig.add_axes([0.905, 0.10, 0.025, 0.78])   # [left, bottom, w, h]
norm    = Normalize(vmin=0, vmax=1)
sm      = cm.ScalarMappable(cmap=cm.get_cmap('turbo_r'), norm=norm)  # _r = flipped
sm.set_array([])
cb = fig.colorbar(sm, cax=cbar_ax)
cb.set_ticks([0, 0.5, 1.0])
cb.set_ticklabels(['Low', 'Mid', 'High'], fontsize=8, fontfamily='serif')
cb.ax.set_ylabel('TF-IDF Rank', fontsize=8, fontfamily='serif', labelpad=6)
cb.outline.set_linewidth(0.5)

fig.suptitle('Word Clouds per Macro-Topic  (TF-IDF distinctive terms)',
             fontsize=13, fontweight='bold', y=0.98, fontfamily='serif')

out_png = OUTPUT_DIR / 'macro_topic_wordclouds.png'
out_pdf = OUTPUT_DIR / 'macro_topic_wordclouds.pdf'
plt.savefig(out_png, dpi=150, bbox_inches='tight', facecolor='white')
plt.savefig(out_pdf, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {out_png}')
print(f'Saved -> {out_pdf}')

## 20. Annotated Temporal Spotlight — Real-World Events

Overlays known Sri Lankan political events on the temporal topic chart. This is the thesis "wow" result: unsupervised NLP discovered real events without being told about them.

In [ ]:
# ── Annotated temporal evolution: most volatile macro-topic per algo ──────────
import matplotlib.ticker as mticker

# Known Sri Lankan political events (year → label)
SL_EVENTS = {
    2019: 'Easter\nBombing',
    2020: 'COVID-19\nLockdowns',
    2022: 'Economic\nCrisis /\nAragalaya',
    2024: 'Presidential\nElection',
}

years_arr = df['year'].values

def volatile_topic_data(labels, kw_dict):
    """Return (x_vals, y_vals, label) for the most year-volatile topic."""
    df_t        = pd.DataFrame({'cluster': labels, 'year': years_arr})
    df_t        = df_t[df_t['cluster'] != -1]
    pivot       = df_t.groupby(['cluster', 'year']).size().unstack(fill_value=0)
    year_totals = df_t.groupby('year').size()
    pct         = pivot.div(year_totals, axis=1) * 100
    vol_id      = pct.std(axis=1).idxmax()
    words       = kw_dict.get(vol_id, [])[:5]
    label       = f'C{vol_id}: {", ".join(words)}'
    x_vals      = sorted(pct.columns)
    y_vals      = [pct.loc[vol_id].get(yr, 0) for yr in x_vals]
    return x_vals, y_vals, label


ALGO_CONFIGS = [
    ('HDBSCAN',       hdb_labels, kw_hdb),
    ('KMeans',        km_labels,  kw_km),
    ('Agglomerative', agg_labels, kw_agg),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (algo_name, labels, kw_dict) in zip(axes, ALGO_CONFIGS):
    x_vals, y_vals, label = volatile_topic_data(labels, kw_dict)

    ax.fill_between(x_vals, y_vals, alpha=0.2, color='steelblue')
    ax.plot(x_vals, y_vals, marker='o', linewidth=2.2,
            color='steelblue', label=label, zorder=3)

    # Annotate real-world events
    for yr, event_label in SL_EVENTS.items():
        if yr in x_vals:
            yr_idx  = x_vals.index(yr)
            yr_val  = y_vals[yr_idx]
            ax.axvline(yr, color='crimson', linewidth=1.2,
                       linestyle='--', alpha=0.7, zorder=2)
            ax.annotate(
                event_label,
                xy=(yr, yr_val),
                xytext=(yr + 0.1, yr_val + 0.5),
                fontsize=7, color='crimson',
                arrowprops=dict(arrowstyle='->', color='crimson', lw=0.8),
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7, lw=0.5),
            )

    ax.set_title(f'{algo_name}\n{label}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Year', fontsize=8)
    ax.set_ylabel('% of yearly speeches', fontsize=8)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(axis='y', linestyle='--', alpha=0.35)

plt.suptitle(
    'Most Volatile Topic per Algorithm — Overlaid with Sri Lankan Political Events\n'
    'Unsupervised NLP discovery: no event labels were provided to the model',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_spotlight_annotated.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> temporal_spotlight_annotated.png')


## 21. Three-Way Model Comparison Table

Table 5.1 in the thesis Results chapter: LDA vs BERTopic vs BiTopic across all metrics.

In [ ]:
# ── Build the master comparison table (Table 5.1) ────────────────────────────
# Loads available metric files and combines with current run's metrics.

comparison_rows = []

# ── Row 1: LDA baseline (from v4/v5 outputs) ─────────────────────────────────
lda_metric_paths = sorted((OUTPUT_DIR.parent).glob('lda_v*/**/lda_metrics.json'))
if lda_metric_paths:
    with open(lda_metric_paths[-1]) as f:
        lda_m = json.load(f)
    comparison_rows.append({
        'Model'        : 'LDA (Baseline)',
        'Architecture' : 'Bag-of-Words + Dirichlet',
        'K (topics)'   : lda_m.get('n_topics', '—'),
        'Silhouette'   : '—',          # LDA does not produce geometric embeddings
        'Coherence c_v': lda_m.get('coherence_cv', '—'),
        'Noise %'      : '0.0',        # LDA assigns every document to a topic
        'Clustered %'  : '100.0',
    })
else:
    comparison_rows.append({
        'Model': 'LDA (Baseline)', 'Architecture': 'Bag-of-Words + Dirichlet',
        'K (topics)': '—', 'Silhouette': '—',
        'Coherence c_v': '(run lda notebook)', 'Noise %': '0.0', 'Clustered %': '100.0',
    })

# ── Row 2: BERTopic / HDBSCAN (current notebook) ─────────────────────────────
N = len(hdb_labels)
noise_hdb = int((np.array(hdb_labels) == -1).sum())
comparison_rows.append({
    'Model'        : 'BERTopic (Semantic)',
    'Architecture' : 'BGE-M3 → UMAP → HDBSCAN',
    'K (topics)'   : K_hdbscan,
    'Silhouette'   : round(float(silhouette_stratified(emb5, hdb_labels)), 4),
    'Coherence c_v': round(cv_micro, 4) if 'cv_micro' in dir() else '(run §15)',
    'Noise %'      : round(100 * noise_hdb / N, 2),
    'Clustered %'  : round(100 * (N - noise_hdb) / N, 2),
})

# ── Row 3: BiTopic (early fusion, from v13 CSV) ───────────────────────────────
v13_csv = OUTPUT_DIR.parent / 'v13_bitopic' / 'v13_comparison.csv'
if v13_csv.exists():
    v13_df = pd.read_csv(v13_csv)
    ef_row = v13_df[v13_df['pipeline'].str.contains('early', case=False)]
    if not ef_row.empty:
        r = ef_row.iloc[0]
        comparison_rows.append({
            'Model'        : 'BiTopic (Early Fusion)',
            'Architecture' : 'BGE-M3 + CountVec → UMAP → HDBSCAN',
            'K (topics)'   : int(r['n_topics']),
            'Silhouette'   : '—',
            'Coherence c_v': '(run v13 §15)',
            'Noise %'      : float(r['noise_pct']),
            'Clustered %'  : float(r['clustered_pct']),
        })

# ── Row 4: Macro-Topic layer ──────────────────────────────────────────────────
macro_clean = df[~df['macro_topic'].isin(['Procedural Noise','Niche/Localized Debates'])]
comparison_rows.append({
    'Model'        : 'Macro-Topic (Final)',
    'Architecture' : 'BERTopic → Centroid Merging → 15 Topics',
    'K (topics)'   : 15,
    'Silhouette'   : '—',
    'Coherence c_v': round(cv_macro, 4) if 'cv_macro' in dir() else '(run §15)',
    'Noise %'      : round(100 * df['macro_topic'].isin(['Procedural Noise']).sum() / N, 2),
    'Clustered %'  : round(100 * len(macro_clean) / N, 2),
})

comp_df = pd.DataFrame(comparison_rows)
print('═' * 80)
print('  TABLE 5.1 — Model Comparison Summary')
print('═' * 80)
print(comp_df.to_string(index=False))
print()

# Save
comp_df.to_csv(OUTPUT_DIR / 'table_5_1_model_comparison.csv', index=False)
print('Saved -> table_5_1_model_comparison.csv')


### Macro-Topic Temporal Evolution (No Buttons)
Static multi-line chart of macro-topic prevalence by year (% of yearly speeches).

In [ ]:
# ── Macro-topic temporal evolution (Raw Counts) ──────
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Full macro-topic labels provided for Chapter 5 reporting
MT_LABELS = {
    0:  'Legislative Mechanics & Standing Orders',
    1:  'Energy Security & Macroeconomic Crisis',
    2:  'National Security, Ethno-Religious Tensions & Law Enforcement',
    3:  'Mega-Infrastructure, Ports & Aviation Development',
    4:  'Agricultural Subsidies, Food Security & Price Controls',
    5:  'Public Welfare, Housing & Water Sanitation',
    6:  'State Enterprise Accountability & COPE Oversight',
    7:  'Educational Administration & Human Capital Development',
    8:  'Constitutional Reform & Executive Powers',
    9:  'Parliamentary Privilege & Member Conduct',
    10: 'Rural Debt Crisis, Microfinance & Cooperatives',
    11: 'Public Healthcare System & Pharmaceutical Regulation',
    12: 'Public Utilities & Media Broadcasting Regulation',
    13: 'Foreign Policy, Peacebuilding & Transitional Justice', 
    14: 'Environmental Conservation & Land Rights Disputes',
    15: 'State Tributes & Parliamentary Condolences',
    16: 'Partisan Politics & Adjournment Resolutions',
    17: 'Easter Sunday Attacks & State Intelligence',
    18: 'Road Infrastructure & Public Transport Networks',
    19: 'Migrant Labour, Tourism & Foreign Exchange',
    20: 'International Relations & Human Rights Frameworks',
    21: 'Livestock Development & Dairy Production',
    22: 'Private Higher Education (SAITM) & University Protests',
    23: 'Local Government Governance & Electoral Reforms',
    24: 'State Revenue, Excise Policy & Gem Industry',
    25: 'National Sports Administration & Cricket Board Crises',
    26: 'Judicial System, Anti-Corruption & Criminal Law',
    27: 'Maritime Boundaries & Fisheries Livelihoods',
    28: 'Plantation Economy & Estate Worker Grievances',
    29: 'Women & Child Affairs, Heritage & Local Governance'
}

# Event labels shown on the timeline
EVENT_LABELS = {
    2019: 'Easter Attacks · Elections',
    2020: 'COVID-19 · 20th Amendment',
    2021: 'Fertilizer Ban · Downturn',
    2022: 'Economic Crisis · Aragalaya',
    2024: 'Presidential · Parl. Elections',
}

# Keep only valid macro-topic rows (exclude procedural/noise-like buckets)
plot_df = df.copy()
plot_df = plot_df[~plot_df['macro_topic'].isin(['Procedural Noise', 'Niche/Localized Debates'])].copy()

# Parse topic ID from strings like "Macro-Topic 7" or "MT-7"
plot_df['mt_id'] = (
    plot_df['macro_topic']
    .astype(str)
    .str.extract(r'(?:Macro-Topic\s*|MT-)(\d+)', expand=False)
)
plot_df = plot_df[plot_df['mt_id'].notna()].copy()
plot_df['mt_id'] = plot_df['mt_id'].astype(int)
plot_df = plot_df[plot_df['mt_id'].between(0, 29)]

# --- CHANGED SECTION: ONLY RAW COUNTS ---
# Get the raw count of speeches per year per topic
year_topic_counts = (
    plot_df.groupby(['year', 'mt_id'])
    .size()
    .reset_index(name='count')
)

# Pivot directly on the 'count' instead of percentages
pivot_counts = (
    year_topic_counts
    .pivot(index='year', columns='mt_id', values='count')
    .fillna(0)
    .sort_index()
)

years = pivot_counts.index.tolist()

# 1. INCREASE LENGTH, DECREASE HEIGHT
fig, ax = plt.subplots(figsize=(24, 8)) 

# 2. CREATE 30 UNIQUE COLORS (combining tab20 and the first 10 of tab20b)
colors = list(plt.cm.tab20.colors) + list(plt.cm.tab20b.colors)[:10]

# 3. ADD LINE STYLES TO FURTHER DIFFERENTIATE
line_styles = ['-', '--', '-.', ':'] 

# Plot the raw counts
for i, mt_id in enumerate(sorted([c for c in pivot_counts.columns if c in MT_LABELS])):
    y = pivot_counts[mt_id].values
    ax.plot(
        years,
        y,
        linewidth=2.0,  # Made slightly thicker for visibility
        alpha=0.85,     # Increased opacity
        color=colors[i],
        linestyle=line_styles[i % 4], # Cycles through solid, dash, dash-dot, dot
        marker='o',                   # Adds a distinct dot at each data point
        markersize=4,
        label=f"MT-{mt_id}: {MT_LABELS[mt_id]}",
    )

ax.set_title('Macro-Topic Temporal Evolution (2017-2026)', fontsize=15, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Number of Speeches', fontsize=12) 

ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_ylim(bottom=0)
ax.grid(True, linestyle='--', alpha=0.3)

# Event guide lines and labels
for yr, lbl in EVENT_LABELS.items():
    if yr in years:
        ax.axvline(yr, color='gray', linestyle='--', linewidth=1.2, alpha=0.5)
        ax.text(yr, -0.05, lbl, transform=ax.get_xaxis_transform(),
                ha='center', va='top', fontsize=10, color='black', fontweight='bold')

# Adjust legend to stretch across the wider bottom
ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=5, # Increased columns to fit the wider aspect ratio
    fontsize=9,
    frameon=False,
)

plt.tight_layout()
plt.savefig('macro_topic_temporal_evolution_counts.png', dpi=300, bbox_inches='tight') # Bumped DPI for print quality
plt.show()
print('Saved -> macro_topic_temporal_evolution_counts.png')

In [ ]:
import numpy as np

# --- ULTRA-WIDE MASTER CHART ---
# Ensure your pivot_counts dataframe is already created from the previous steps

years = pivot_counts.index.tolist()
mt_ids = sorted([c for c in pivot_counts.columns if c in MT_LABELS])

# 1. ULTRA-WIDE Canvas: Increased width to 32
fig, ax = plt.subplots(figsize=(32, 14)) 

# 2. High-Contrast Colors: Sample 30 evenly spaced colors from the vibrant 'turbo' map
cmap = plt.get_cmap('turbo')
colors = [cmap(i) for i in np.linspace(0.05, 0.95, len(mt_ids))]

# 3. Line styles and markers for differentiation
line_styles = ['-', '--', '-.', ':']
markers = ['o', 's', '^', 'D', 'v']

# Plot the raw counts
for i, mt_id in enumerate(mt_ids):
    y = pivot_counts[mt_id].values
    ax.plot(
        years,
        y,
        linewidth=2.5,
        alpha=0.85,
        color=colors[i],
        linestyle=line_styles[i % 4],
        marker=markers[i % 5],
        markersize=6,
        label=f"MT-{mt_id}: {MT_LABELS[mt_id]}"
    )

# Formatting Aesthetics
ax.set_title('Macro-Topic Temporal Evolution (2017-2026)', fontsize=24, fontweight='bold', pad=20)
ax.set_xlabel('Year', fontsize=16, fontweight='bold', labelpad=15)
ax.set_ylabel('Number of Speeches', fontsize=16, fontweight='bold', labelpad=15) 

ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.tick_params(axis='both', which='major', labelsize=14)
ax.set_ylim(bottom=0)

# Clean grid
ax.grid(True, linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 4. FIXED EVENT LABELS: Placed at the top with protective background boxes
for i, (yr, lbl) in enumerate(EVENT_LABELS.items()):
    if yr in years:
        ax.axvline(yr, color='#555555', linestyle='--', linewidth=1.5, alpha=0.6)
        
        # Alternate the vertical position slightly so close years don't overlap
        y_pos = 0.98 - (i % 2) * 0.04 
        
        ax.text(
            yr + 0.05, y_pos, lbl, 
            transform=ax.get_xaxis_transform(),
            ha='left', va='top', 
            fontsize=14, color='black', fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.85, edgecolor='lightgray', pad=4) 
        )

# 5. LEGEND: 5 columns, pushed lower with adjusted text sizing
ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.12), 
    ncol=5,                      
    fontsize=12,                 
    frameon=False,
    labelspacing=1.2             
)

plt.tight_layout()
plt.savefig('macro_topic_temporal_evolution_ultrawide.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved -> macro_topic_temporal_evolution_ultrawide.png')

In [ ]:
# FROM:
kw_out = {
    'count' : {str(k): [w for w, _ in v] for k, v in kw_macro_count.items()},
    'tfidf' : {str(k): [w for w, _ in v] for k, v in kw_macro_tfidf.items()},
}
kw_path = OUTPUT_DIR / 'macro_topic_keywords.json'
with open(kw_path, 'w', encoding='utf-8') as f:
    _json.dump(kw_out, f, ensure_ascii=False, indent=2)

# TO:
kw_out = {
    'count' : {str(k): [w for w, _ in v] for k, v in kw_macro_count.items()},
    'tfidf' : {str(k): [w for w, _ in v] for k, v in kw_macro_tfidf.items()},
    'count_with_freq': {str(k): [{'keyword': w, 'count': c} for w, c in v] for k, v in kw_macro_count.items()},
}
kw_path = OUTPUT_DIR / 'macro_topic_keywords_100.json'
with open(kw_path, 'w', encoding='utf-8') as f:
    _json.dump(kw_out, f, ensure_ascii=False, indent=2)
